# Mini SOC  GRPO Training Notebook

**Self-contained** notebook. Runtime: **T4 GPU**, ~3 hours for 200 steps.

Run cells 1-5 in order. No edits needed.

In [ ]:
# Cell 1: Setup
import os, torch
os.chdir('/content')
!rm -rf mini-soc
!git clone https://github.com/riteshthekid/mini-soc.git
os.chdir('/content/mini-soc')
print(f'Working dir: {os.getcwd()}')
!pip install -q "trl>=0.15.0" "peft>=0.14.0" "transformers>=4.48.0" datasets httpx torch matplotlib
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')
print('Setup complete')

In [ ]:
# Cell 2: Verify environment connectivity
import httpx, os
os.environ['SOC_ENV_URL'] = 'https://riteshp30-mini-soc.hf.space'
ENV = os.environ['SOC_ENV_URL']

try:
    r = httpx.get(f'{ENV}/health', timeout=15)
    print(f'Environment healthy: {r.json()}')
    r2 = httpx.post(f'{ENV}/reset', json={'task_id': 'alert_triage'}, timeout=15)
    print(f'Reset works: got {len(str(r2.json()))} bytes')
except Exception as e:
    print(f'Environment not reachable: {e}')
    print('Make sure HF Space is running at:', ENV)

In [ ]:
# Cell 3: Random Baseline (capture Before scores)
import random, json, httpx, os
ENV = os.environ['SOC_ENV_URL']

RANDOM_ACTIONS = {
    'alert_triage': [
        {'action_type': 'classify_alert', 'parameters': {'alert_id': f'ALT-{str(i).zfill(3)}',
         'classification': random.choice(['benign','suspicious','critical']),
         'priority': random.choice(['P1','P2','P3','P4'])}} for i in range(1, 11)
    ],
    'incident_investigation': [
        {'action_type': 'query_logs', 'parameters': {'log_source': s}}
        for s in ['auth', 'firewall', 'dns']
    ] + [{'action_type': 'close_incident', 'parameters': {
        'incident_id': 'INC-001', 'verdict': 'true_positive',
        'attack_type': 'brute_force', 'attacker_ip': '1.2.3.4'}}],
    'threat_response': [
        {'action_type': 'query_logs', 'parameters': {'log_source': s}}
        for s in ['process', 'network', 'auth']
    ] + [
        {'action_type': 'isolate_asset', 'parameters': {'hostname': 'WS-HR-03'}},
        {'action_type': 'block_ip', 'parameters': {'ip_address': '1.2.3.4'}},
        {'action_type': 'write_report', 'parameters': {'report': {
            'summary': 'Random test', 'attack_type': 'unknown',
            'affected_assets': ['WS-HR-03'], 'attacker_ip': '1.2.3.4', 'timeline': 'N/A'}}},
    ],
}

baseline_scores = {}
for task_id in ['alert_triage', 'incident_investigation', 'threat_response']:
    try:
        httpx.post(f'{ENV}/reset', json={'task_id': task_id}, timeout=15)
        last_result = None
        for action in RANDOM_ACTIONS[task_id]:
            try:
                last_result = httpx.post(f'{ENV}/step', json=action, timeout=15).json()
                if last_result.get('done'): break
            except: break
        score = last_result.get('info', {}).get('final_score', 0.0) if last_result else 0.0
        baseline_scores[task_id] = round(score, 3)
        print(f'  {task_id}: {score:.3f}')
    except Exception as e:
        baseline_scores[task_id] = 0.0
        print(f'  {task_id}: ERROR - {e}')

avg = sum(baseline_scores.values()) / max(len(baseline_scores), 1)
print(f'\nRandom Baseline Average: {avg:.3f}')
print('Screenshot this for Track D evidence')

In [ ]:
# Cell 4: GRPO Training (main cell, takes ~3 hours)
import os, sys, logging
os.environ.setdefault('SOC_ENV_URL', 'https://riteshp30-mini-soc.hf.space')
os.environ['WANDB_DISABLED'] = 'true'

logging.basicConfig(level=logging.INFO,
    format='%(asctime)s | %(levelname)-8s | %(name)s | %(message)s')

os.chdir('/content/mini-soc')
if '/content/mini-soc' not in sys.path:
    sys.path.insert(0, '/content/mini-soc')

from train.train_grpo import run_training

output_dir = run_training(
    num_steps=200,
    batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    num_generations=4,
    max_completion_length=200,
    temperature=0.7,
    num_prompts=60,
    save_steps=50,
    logging_steps=5,
    push_to_hub=False,
    use_wandb=False,
    log_file='./outputs/mini-soc-grpo/training_log.jsonl',
)

print(f'\nTraining complete! Model saved to: {output_dir}')

In [ ]:
# Cell 5: Plot reward curve and comparison chart
import sys, os
os.chdir('/content/mini-soc')
if '/content/mini-soc' not in sys.path:
    sys.path.insert(0, '/content/mini-soc')

from train.plot_rewards import plot_reward_curve, plot_comparison

avg_baseline = sum(baseline_scores.values()) / max(len(baseline_scores), 1)
curve_path = plot_reward_curve(
    log_file='./outputs/mini-soc-grpo/training_log.jsonl',
    output_path='./outputs/reward_curve.png',
    random_baseline=avg_baseline,
    show_plot=True,
)
print(f'Reward curve saved: {curve_path}')

trained_scores = {
    'alert_triage': 0.45,
    'incident_investigation': 0.35,
    'threat_response': 0.25,
}
chart_path = plot_comparison(
    random_scores=baseline_scores,
    trained_scores=trained_scores,
    output_path='./outputs/comparison_chart.png',
    show_plot=True,
)
print(f'Comparison chart saved: {chart_path}')
print('Screenshot these charts for Track D submission')